### Datos Sintéticos
### Librería SDV
*Fecha: 05 de junio*

*Autor: Maria Reina Zarate*

In [1]:
# Importamos librerías básicas
import pandas as pd
import numpy as np
import random

In [2]:
# Importamos librerías para datos sintéticos
import sdv
import sdmetrics

In [3]:
# Importamos sublibrerías para la generación de datos
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [4]:
# Importamos sublibrerias para la evaluación de los datos
from sdmetrics.reports.single_table import QualityReport

In [5]:
print("SDV: ", sdv.__version__)
print("SDMetrics: ", sdmetrics.__version__)

SDV:  1.37.0
SDMetrics:  0.28.0


In [6]:
# Crear una tabla de datos REAL como base para la 
# generación de Datos Sintéticos.
dfClientes = pd.DataFrame(
    {
        "cliente_id" : [1,2,3,4,5,6,7,8,9,10],
        "edad" : [17,18,19,25,32,42,70,28,23,22],
        "ingreso_mensual" : [1800,2000,2500,15000,18000,35000,7000,15000,4000,5000],
        "ciudad": ["Potrero Nuevo", "Atoyac","Cuichapa","Córdoba","Cuitáhuac", "Orizaba", "Paraje","Yanga","Tierra Blanca","Calcahualco"]
    }
)

In [7]:
dfClientes

,cliente_id,edad,ingreso_mensual,ciudad
0,1,17,1800,Potrero Nuevo
1,2,18,2000,Atoyac
2,3,19,2500,Cuichapa
3,4,25,15000,Córdoba
4,5,32,18000,Cuitáhuac
5,6,42,35000,Orizaba
6,7,70,7000,Paraje
7,8,28,15000,Yanga
8,9,23,4000,Tierra Blanca
9,10,22,5000,Calcahualco


### Obtener los metadata del DataFrame generado

In [10]:
# Creamos una instancia de SingleTableMetadata
metadata = SingleTableMetadata()

In [11]:
metadata.detect_from_dataframe(
    data= dfClientes
)

In [12]:
metadata.to_dict()

{'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}},
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'primary_key': 'cliente_id'}

metadata.visualize()

### Entrenar el modelo de datos sintéticos
*Un sinthetisizer es el encargado de generar los datos sintéticos*

In [15]:
# Le pasamos los metadatos
sintetizador =  GaussianCopulaSynthesizer(
    metadata
)

/opt/anaconda3/envs/Extraccion/lib/python3.11/site-packages/sdv/single_table/base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/opt/anaconda3/envs/Extraccion/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [16]:
# Entrenamos el sintetizador con los datos "REALES"
sintetizador.fit(
    dfClientes
)

In [17]:
#Generamos los datos sintéticos
dfClientes_falsos = sintetizador.sample(
    num_rows=200
)

In [19]:
dfClientes_falsos

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,36,26060,Paraje
1,4918803,20,4071,Orizaba
2,1900081,20,6892,Córdoba
3,531516,20,2864,Cuichapa
4,10211768,38,15528,Paraje
...,...,...,...,...
195,1577423,18,5283,Atoyac
196,2804584,18,2059,Potrero Nuevo
197,10420935,36,10666,Yanga
198,16659394,22,8308,Yanga


In [20]:
dfClientes_falsos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   cliente_id       200 non-null    int64 
 1   edad             200 non-null    int64 
 2   ingreso_mensual  200 non-null    int64 
 3   ciudad           200 non-null    object
dtypes: int64(3), object(1)
memory usage: 6.4+ KB


In [21]:
dfClientes.describe()

,cliente_id,edad,ingreso_mensual
count,10.00000,10.000000,10.000000
mean,5.50000,29.600000,10530.000000
std,3.02765,16.063762,10507.568701
min,1.00000,17.000000,1800.000000
25%,3.25000,19.750000,2875.000000
50%,5.50000,24.000000,6000.000000
75%,7.75000,31.000000,15000.000000
max,10.00000,70.000000,35000.000000


In [22]:
dfClientes_falsos.describe()

,cliente_id,edad,ingreso_mensual
count,2.000000e+02,200.000000,200.000000
mean,8.765887e+06,26.145000,10636.865000
std,4.707538e+06,12.724525,9092.254004
min,4.660500e+04,18.000000,2000.000000
25%,5.085226e+06,18.000000,2691.250000
50%,9.212590e+06,20.000000,7996.000000
75%,1.279456e+07,29.000000,15688.250000
max,1.677377e+07,70.000000,35000.000000


### Evaluamos los datos para validar
1. Similitud entre las columnas (Columns Shapes Score)
2. Similitud entre las relaciones de las columnas (Evaluating Column Per Trends)
3. Evaluación del dataset general

In [24]:
#Evaluamos la calidad de los datos
#Creamos una instancia de QualityReport
reporte = QualityReport()

### Puntos de referencia

|Score   | Interpretación |
|--------|----------------|
|1   | Indenticas |
|0.9 | Muy similar |
|0.7 | Similar |
|0.6 | Diferencias visibles|
|< 0.5 | Muy diferentes |

In [25]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data =  dfClientes_falsos,
    metadata =  metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |███████████████| 4/4 [00:00<00:00, 644.66it/s]|
Column Shapes Score: 79.5%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 282.90it/s]|
Column Pair Trends Score: 33.75%

Overall Score (Average): 56.62%

